In [1]:
from pathlib import Path

from maite_datasets.image_classification import MilitaryVehicles

data_root = Path("./data")

# One download shared by every export; a re-run reads what is already on disk.
MilitaryVehicles(root=data_root, image_set="base", download=True)
MilitaryVehicles(root=data_root, image_set="train", as_datamaite=True)

# The export nests its images one level down, under the split name.
data_path = data_root / "militaryvehicles_datamaite_train" / "train"
print(f"Reading from {data_path}")

/home/aweng/2033/dataeval-flow/.nox/docs/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Reading from data/militaryvehicles_datamaite_train/train


In [2]:
from dataeval_flow import PipelineConfig, run_task
from dataeval_flow.config import HuggingFaceDatasetConfig, SourceConfig, TaskConfig
from dataeval_flow.workflows.data_splitting import DataSplittingConfig

workflow = DataSplittingConfig(
    name="mv_split",
    test_frac=0.2,  # 20% of full dataset held out for test
    folds=3,  # 3-fold cross-validation; each fold's val is 1/3 of the non-test items
    stratify=True,  # Preserve class distribution in each partition
)

task = TaskConfig(
    name="split_military_vehicles",
    workflow="mv_split",
    sources="mv_src",
)

# Build the pipeline configuration
config = PipelineConfig(
    datasets=[
        HuggingFaceDatasetConfig(name="mv_train", path=str(data_path), task="image_classification"),
    ],
    sources=[
        SourceConfig(name="mv_src", dataset="mv_train"),
    ],
    workflows=[workflow],
    tasks=[task],
)

In [3]:
result = run_task(task, config, cache_dir=Path("./cache"))

/home/aweng/2033/dataeval-flow/src/dataeval_flow/evaluators/bias/_evaluator.py:50: UserWarning: `height` and `width` were binned automatically ('uniform_width') because no bins were declared. The bin count is derived from the data, so it is not stable across samples and the same factor measured twice may not be comparable. Declare cutoffs with continuous_factor_bins={"height": [...]} to control this.
  return Balance(**balance_arguments(config, source.metadata_policy)).evaluate(metadata)


In [4]:
if not result.success:
    print(f"Workflow failed: {result.errors}")
assert result.success

In [5]:
print(result.report())


  DATA SPLITTING
  Workflow:  mv_split (data-splitting)
  Timestamp: 2026-10-03T05:05:09.694235+00:00
  Duration:  0.26s
  Source:    mv_src (mv_train)

  Steps: 13 (9 ran, 4 skipped)

  SUMMARY
  Label Distribution ........... 24 classes, 7823 items, imbalance 3.6:1  [..]
  0
    Stratification .... max deviation 0.1pp (class 'T-72' in split.test)  [ok]
  1
    Stratification .... max deviation 0.1pp (class 'T-72' in split.test)  [ok]
  2
    Stratification .... max deviation 0.1pp (class 'T-72' in split.test)  [ok]

  Health: All checks passed [ok]

  LABEL DISTRIBUTION                     24 classes, 7823 items, imbalance 3.6:1
  Class       Count
  ----------  -----  ------------------------------
  T-72          424  ██████████████████████████████
  BTR-80        413  █████████████████████████████▎
  BMP-1         393  ███████████████████████████▊
  BTR-60        391  ███████████████████████████▋
  BMP-2         390  ███████████████████████████▋
  BRDM          389  █████████████

In [6]:
indices = result.steps["split"].details["indices"]
test = indices["test"]
print(f"Folds: {list(indices['train'])}")
print(f"Test indices: {len(test)} (first ten: {test[:10]})")

Folds: ['0', '1', '2']
Test indices: 1565 (first ten: [69, 70, 71, 72, 73, 74, 75, 76, 77, 78])


In [7]:
import polars as pl

rows = [
    {"fold": fold, "train": len(indices["train"][fold]), "val": len(indices["val"][fold])} for fold in indices["train"]
]
print(pl.DataFrame(rows))
print(f"\nTest (shared across folds): {len(test)} samples")

shape: (3, 3)
┌──────┬───────┬──────┐
│ fold ┆ train ┆ val  │
│ ---  ┆ ---   ┆ ---  │
│ str  ┆ i64   ┆ i64  │
╞══════╪═══════╪══════╡
│ 0    ┆ 4172  ┆ 2086 │
│ 1    ┆ 4172  ┆ 2086 │
│ 2    ┆ 4172  ┆ 2086 │
└──────┴───────┴──────┘

Test (shared across folds): 1565 samples


In [8]:
# Verify no overlap between parts and full coverage per fold
dataset_size = sum(len(indices[part]["0"]) for part in ("train", "val")) + len(test)
test_set = set(test)

for fold in indices["train"]:
    train_set = set(indices["train"][fold])
    val_set = set(indices["val"][fold])

    assert train_set.isdisjoint(val_set), f"Fold {fold}: train/val overlap!"
    assert train_set.isdisjoint(test_set), f"Fold {fold}: train/test overlap!"
    assert val_set.isdisjoint(test_set), f"Fold {fold}: val/test overlap!"
    assert len(train_set | val_set | test_set) == dataset_size, f"Fold {fold}: missing indices"

print(f"All {len(indices['train'])} folds verified: no overlap, full coverage of {dataset_size} items.")

All 3 folds verified: no overlap, full coverage of 7823 items.


In [9]:
for item in result.findings:
    print(f"{item.severity:8} {item.title}: {item.brief}")

info     Label Distribution: 24 classes, 7823 items, imbalance 3.6:1
ok       Stratification: max deviation 0.1pp (class 'T-72' in split.test)
ok       Stratification: max deviation 0.1pp (class 'T-72' in split.test)
ok       Stratification: max deviation 0.1pp (class 'T-72' in split.test)


In [10]:
print(result.steps["balance"].output.balance)
print(result.steps["diversity"].output.factors)

shape: (3, 2)
┌─────────────┬──────────┐
│ factor_name ┆ mi_value │
│ ---         ┆ ---      │
│ cat         ┆ f64      │
╞═════════════╪══════════╡
│ class_label ┆ 1.0      │
│ height      ┆ 0.008153 │
│ width       ┆ 0.009517 │
└─────────────┴──────────┘
shape: (3, 3)
┌─────────────┬─────────────────┬──────────────────┐
│ factor_name ┆ diversity_value ┆ is_low_diversity │
│ ---         ┆ ---             ┆ ---              │
│ cat         ┆ f64             ┆ bool             │
╞═════════════╪═════════════════╪══════════════════╡
│ class_label ┆ 0.957973        ┆ false            │
│ height      ┆ 0.153298        ┆ true             │
│ width       ┆ 0.169882        ┆ true             │
└─────────────┴─────────────────┴──────────────────┘


In [11]:
print(f"Folds:      {len(indices['train'])}")
print(f"Source:     {result.metadata.lineage[0].name} ({result.metadata.lineage[0].items} items)")
for record in result.metadata.lineage[1:4]:
    print(f"{record.name:16} {record.type:6} {record.items} items")

Folds:      3
Source:     data (7823 items)
split.train[0]   kfold  4172 items
split.train[1]   kfold  4172 items
split.train[2]   kfold  4172 items


In [12]:
import json

exported = json.loads(result.export(fmt="json"))

# The same indices, under the split step's details
exported_indices = exported["steps"]["split"]["details"]["indices"]
print(f"Test indices ({len(exported_indices['test'])} samples): {exported_indices['test'][:10]}...")

for fold in exported_indices["train"]:
    print(f"Fold {fold}: train={len(exported_indices['train'][fold])}, val={len(exported_indices['val'][fold])}")

Test indices (1565 samples): [69, 70, 71, 72, 73, 74, 75, 76, 77, 78]...
Fold 0: train=4172, val=2086
Fold 1: train=4172, val=2086
Fold 2: train=4172, val=2086
